In [ ]:
# ============================================
# 0. Install dependencies (Colab)
# ============================================
!pip install -q transformers torch tqdm




In [ ]:
# ============================================
# 1. Imports & model loading
# ============================================
import json
from collections import Counter
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "cardiffnlp/twitter-roberta-base-emotion-latest"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

id2label = model.config.id2label  # e.g. {0: 'anger', 1: 'anticipation', ...}
all_labels = set(id2label.values())

print("Using device:", device)
print("Emotion labels:", id2label)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Using device: cpu
Emotion labels: {0: 'anger', 1: 'anticipation', 2: 'disgust', 3: 'fear', 4: 'joy', 5: 'love', 6: 'optimism', 7: 'pessimism', 8: 'sadness', 9: 'surprise', 10: 'trust'}


In [ ]:
# ============================================
# 2. Define polarity groups
# ============================================
NEG_EMOS = {"anger", "disgust", "fear", "pessimism", "sadness"}
POS_EMOS = {"joy", "love", "optimism", "trust", "anticipation"}
# We treat NEUTRAL separately → created by rule





In [ ]:
# ============================================
# 3. Rule-based neutral detection
# ============================================
def detect_neutral(probs, threshold=0.30):
    """
    If the max probability of any emotion is below threshold,
    we consider the case neutral.
    """
    if probs.max() < threshold:
        return True
    return False




In [ ]:
# ============================================
# 4. Predict a single emotion OR neutral
# ============================================
def predict_emotion_single(text, neutral_threshold=0.30):
    """
    Returns: one emotion OR "neutral"
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.sigmoid(logits)[0].cpu().numpy()

    # Check neutral condition first
    if detect_neutral(probs, threshold=neutral_threshold):
        return "neutral"

    # Otherwise, choose strongest emotion
    best_idx = int(probs.argmax())
    best_label = id2label[best_idx]
    return best_label

In [ ]:

# ============================================
# 5. Polarity filtering including neutral
# ============================================
def filter_by_polarity(emotion, polarity):
    """
    - If neutral → keep neutral.
    - If matches polarity → keep.
    - If contradicts → keep anyway (we do not override with neutral).
    """
    polarity = polarity.lower().strip()

    if emotion == "neutral":
        return emotion

    if polarity == "positive":
        allowed = POS_EMOS
        return emotion if emotion in allowed else emotion

    if polarity == "negative":
        allowed = NEG_EMOS
        return emotion if emotion in allowed else emotion

    # Neutral polarity → keep whatever the model predicted
    return emotion

In [ ]:

# ============================================
# 6. Process a JSONL dataset
# ============================================
def process_jsonl(input_path, output_path):
    """
    Fills the 'emotion' field (single-label or 'neutral').
    Returns distribution Counter.
    """
    emotion_counter = Counter()
    updated_entries = []

    with open(input_path, "r", encoding="utf-8") as f:
        entries = [json.loads(line) for line in f]

    for entry in tqdm(entries, desc=f"Processing {input_path}"):
        review = entry["input"]

        for item in entry["output"]:
            aspect = item["aspect"]
            polarity = item.get("polarity", "neutral")

            # Improved prompt
            prompt = (
                f"Review: {review}\n"
                f"What emotion does the reviewer express about the {aspect}? "
                f"Choose one emotion word."
            )

            # 1. Predict raw emotion or neutral
            raw_emotion = predict_emotion_single(prompt)

            # 2. Optional polarity filtering
            final_emotion = filter_by_polarity(raw_emotion, polarity)

            # Save result
            item["emotion"] = final_emotion
            emotion_counter[final_emotion] += 1

        updated_entries.append(entry)

    # Save updated file
    with open(output_path, "w", encoding="utf-8") as f:
        for entry in updated_entries:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    # Print distribution
    print("\nEmotion distribution:")
    for emo, cnt in emotion_counter.most_common():
        print(f"{emo}: {cnt}")

    return emotion_counter


# ============================================
# 7. Run on your 300-review dataset
# ============================================
input_file_300 = "/content/annotation_300.jsonl"
output_file_300 = "/content/annotation_300_with_emotions.jsonl"

dist_300 = process_jsonl(input_file_300, output_file_300)

Processing /content/annotation_300.jsonl:   0%|          | 0/296 [00:00<?, ?it/s]


Emotion distribution:
joy: 276
anger: 158
anticipation: 145
sadness: 83
disgust: 20
optimism: 5
surprise: 2
fear: 2


In [ ]:
input_file_train = "/content/train.jsonl"
output_file_train = "/content/train_with_emotions.jsonl"
dist_train = process_jsonl(input_file_train, output_file_train)

Processing /content/train.jsonl:   0%|          | 0/3149 [00:00<?, ?it/s]


Emotion distribution:
joy: 2821
anticipation: 1547
anger: 1329
sadness: 991
disgust: 254
optimism: 72
neutral: 23
surprise: 22
fear: 21
love: 10
